This notebook connects to the Transport NSW Regional Buses API for Sydney Surrounds and feeds incoming data into an Eventstream. The code below builds on the Spark code available in this article - https://medium.com/@francescogiorgio.fava/real-time-ferry-tracking-in-sydney-harbour-with-microsoft-fabric-07bb2784ca50.

In [ ]:
pip install requests pandas azure-servicebus gtfs-realtime-bindings protobuf pytz

In [ ]:
myapikey = "" # Your API key to be added here
myconnectionstring = "" # Your Event Hub connection string to be added here

In [ ]:
import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd
import time
from datetime import datetime
import json
from azure.servicebus import ServiceBusClient, ServiceBusMessage
from typing import Dict, Any, Optional
import pytz  # Added for timezone handling

def get_bus_positions(api_key: str) -> Optional[pd.DataFrame]:
    url = 'https://api.transport.nsw.gov.au/v1/gtfs/vehiclepos/regionbuses/sydneysurrounds'
    headers = {
        'accept': 'application/x-google-protobuf',
        'Authorization': f'apikey {api_key}'
    }
    
    try:
        # Fetch the protobuf data
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        
        # Parse the protobuf message
        feed = gtfs_realtime_pb2.FeedMessage()
        feed.ParseFromString(response.content)
        
        # Extract bus positions
        bus_data = []
        for entity in feed.entity:
            if entity.HasField('vehicle'):
                vehicle = entity.vehicle
                bus_info = {
                    'bus_name': vehicle.vehicle.id,
                    'bus_lat': float(vehicle.position.latitude),
                    'bus_long': float(vehicle.position.longitude),
                    'bus_speed': vehicle.position.speed
                }
                bus_data.append(bus_info)
        
        # Create DataFrame only if we have data
        if bus_data:
            return pd.DataFrame(bus_data)
        else:
            print("No bus data found in the response")
            return pd.DataFrame()
            
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error parsing data: {e}")
        return pd.DataFrame()

def dataframe_to_json_messages(df: pd.DataFrame) -> list[Dict[str, Any]]:
    # Create Sydney timezone object
    sydney_tz = pytz.timezone('Australia/Sydney')
    
    messages = []
    
    for _, row in df.iterrows():
        message = {
            "bus_name": str(row['bus_name']),
            "bus_lat": float(row['bus_lat']),
            "bus_long": float(row['bus_long']),
            "bus_speed": float(row['bus_speed']),
            "timestamp": datetime.now(sydney_tz).isoformat()
        }
        messages.append(message)
    
    return messages

# Rest of the code remains the same as in the original script
def send_to_eventstream(messages: list[Dict[str, Any]], connection_string: str) -> None:
    #Set timezone
    sydney_tz = pytz.timezone('Australia/Sydney')

    # Parse the EntityPath from the connection string
    entity_path = None
    for param in connection_string.split(';'):
        if param.startswith('EntityPath='):
            entity_path = param.split('=')[1]
            break
    
    if not entity_path:
        raise ValueError("EntityPath not found in connection string")
        
    servicebus_client = ServiceBusClient.from_connection_string(connection_string)
    
    try:
        with servicebus_client.get_queue_sender(entity_path) as sender:
            # Create a batch of messages
            batch_message = []
            for msg_dict in messages:
                json_string = json.dumps(msg_dict)
                batch_message.append(ServiceBusMessage(json_string))
            
            # Send the batch of messages
            sender.send_messages(batch_message)
            print(f"{datetime.now(sydney_tz).isoformat()}: Successfully sent {len(messages)} bus positions")
            
    except Exception as e:
        print(f"Error sending messages: {e}")
        print(f"Using queue name: {entity_path}")
    finally:
        servicebus_client.close()

def monitor_bus_positions(api_key: str, connection_string: str, interval_seconds: int = 15):
    print(f"Starting bus position monitoring (interval: {interval_seconds} seconds)")

    while True:
        try:
            # Get bus positions
            bus_df = get_bus_positions(api_key)
            
            # Check if DataFrame exists and has data
            if bus_df is not None and not bus_df.empty:
                messages = dataframe_to_json_messages(bus_df)
                send_to_eventstream(messages, connection_string)
            else:
                print(f"{datetime.now()}: No bus data available")
                
        except Exception as e:
            print(f"Error in monitoring loop: {e}")
            
        # Wait for next interval
        time.sleep(interval_seconds)

In [ ]:
if __name__ == "__main__":
    # Configuration
    API_KEY = myapikey
    
    # Make sure the EntityPath in the connection string matches your queue name exactly
    CONNECTION_STRING = myconnectionstring
                       
    INTERVAL_SECONDS = 15
    
    # Start monitoring
    monitor_bus_positions(API_KEY, CONNECTION_STRING, INTERVAL_SECONDS)